# Level 4 -  Multiple Linear Regression

## Objective

The purpose of this analysis is to develop a multiple linear regression model to examine how customer, product, geographic, shipping, and transactional characteristics relate to sales. The dependent variable is **sales**, while the independent variables include a combination of categorical, numerical, and temporal features.

The initial model uses the following predictors:

* **Shipping:** `ship_mode`
* **Customer:** `segment`
* **Geography:** `region`
* **Product Classification:** `sub_category`
* **Transaction:** `quantity`, `discount`
* **Time:** `order_year`, `order_month`

The dependent variable is:

* **Target:** `sales`

### Variable Selection

Variables were selected based on their potential relationship with sales while avoiding identifiers, unnecessary redundancy, and information that could introduce target leakage.

Several variables available in the dataset were intentionally excluded:

* **Identifiers (`row_id`, `order_id`, `customer_id`, `product_id`)** were excluded because they primarily identify individual records, customers, or products rather than represent meaningful explanatory characteristics.

* **Customer and product names (`customer_name`, `product_name`)** were excluded because they are high-cardinality descriptive fields and are not appropriate predictors for this general sales model. Product-level analysis was also avoided because inconsistencies between product IDs and product names were identified during preprocessing.

* **Detailed geographic variables (`city`, `state`, `postal_code`, `country`)** were excluded in favor of `region`. Country provides no useful variation because all observations are from the United States, while city, state, and postal code introduce substantially greater categorical complexity. `region` contains only **4 categories**, requiring **3 dummy variables** when one category is used as the reference group. In comparison, using `state` would require approximately **48 dummy variables**. Region was therefore selected to capture broad geographic differences while keeping the model parsimonious and interpretable.

* **`category`** was excluded in favor of `sub_category`. Although `category` would require only **2 dummy variables** for its 3 product categories, it provides a relatively broad classification of the products being sold. `sub_category` contains **17 categories**, requiring **16 dummy variables** when one category is used as the reference group. This additional complexity was considered reasonable because subcategory captures meaningful differences among product types that would be lost at the broader category level. Including both `category` and `sub_category` was avoided because subcategories are nested within categories and would introduce redundant information.

* **Exact dates (`order_date`, `ship_date`)** were excluded in favor of the derived temporal features `order_year` and `order_month`.

* **`fulfillment_days`** was excluded because it is determined after an order is placed and therefore is not an ideal explanatory variable for the value of the sale itself.

* **`profit` and `profit_margin`** were excluded because they are financially derived from or closely related to sales and could introduce leakage or circular relationships when sales is the prediction target.

* **Customer-level engineered features (`customer_lifetime_sales`, `customer_span_days`, `customer_tier`)** were excluded because they summarize customer behavior across multiple transactions. In particular, customer lifetime sales and customer tier incorporate sales information from the broader dataset and could introduce target leakage into a model predicting individual sales observations.

This feature selection produces a more parsimonious model focused on characteristics that are known at or near the time of the transaction while preserving predictors with clear business interpretations. Categorical variables will be encoded prior to modeling, and model diagnostics will be used to evaluate the assumptions and performance of multiple linear regression.


### Why Choose Sales Over Profit?

Sales was selected as the dependent variable because it represents the direct monetary value of each transaction and can be examined using characteristics available in the dataset, such as product classification, quantity, discount, customer segment, geography, shipping method, and time.

Profit is also an important business outcome, but it reflects both revenue and the costs associated with generating that revenue. Although the dataset provides the final `profit` value for each observation, it does not provide the underlying cost components—such as product cost, shipping expense, or other operating costs—that produced that value. As a result, a model predicting profit would be unable to directly account for several important drivers of profitability.

For this analysis, sales therefore provides a more appropriate and interpretable target for evaluating how the available transaction characteristics relate to monetary performance. Profit remains valuable for descriptive business analysis but is not used as the dependent variable in this regression model.


In [ ]:
X = [
    "ship_mode",
    "segment",
    "region",
    "sub_category",
    "quantity",
    "discount",
    "order_year",
    "order_month"
]

y = "sales"


## Data Preparation for Multiple Linear Regression

Before fitting the regression model, the selected independent variables must be prepared according to their data type. Multiple linear regression requires numerical inputs, so categorical predictors must be converted into numerical form before they can be included in the model.

The selected predictors can be divided into two groups:

**Categorical features:**

* `ship_mode`
* `segment`
* `region`
* `sub_category`
* `order_month`
* `order_year`

**Numerical features:**

* `quantity`
* `discount`

Although `order_year` and `order_month` are stored numerically, both will be treated as categorical variables. Their numerical values primarily identify time periods rather than represent continuous quantities. Treating them as continuous predictors would require the regression model to assume a constant linear change in sales from one year or month to the next. Categorical encoding instead allows individual years and months to have distinct relationships with sales, making fewer assumptions about the temporal pattern in the data.

Categorical variables will be converted to dummy variables using one-hot encoding. One category from each categorical feature will be omitted and used as the reference category. This prevents perfect multicollinearity among the encoded variables and allows the coefficient for each remaining category to be interpreted relative to its corresponding reference group.
